# RNN Encoder–Decoder for Statistical Machine Translation

The **RNN Encoder–Decoder** architecture was introduced to overcome the limitations of traditional RNNs in sequence-to-sequence tasks such as machine translation. Unlike a standard RNN, it can handle input and output sequences of different lengths by using two components: an **Encoder** and a **Decoder**.

# Objective

The main objectives of the RNN Encoder–Decoder are to:

- Handle variable-length input and output sequences.
- Learn meaningful representations of input sentences.
- Improve machine translation accuracy.
- Preserve the meaning of the input while generating the output.
- Overcome the limitations of traditional RNNs.

# Introduction

A **Recurrent Neural Network (RNN)** is designed to process sequential data such as text and speech. It stores information from previous time steps using a **hidden state**, allowing it to understand the context of a sequence.

```text
Current Hidden State =
Previous Hidden State + Current Input
```

Traditional RNNs work well when each input has one corresponding output, such as:

- Part-of-Speech (POS) Tagging
- Named Entity Recognition (NER)
- Video Frame Labeling

# Traditional RNN

A traditional RNN follows a one-to-one relationship between inputs and outputs.

```text
Input 1 → Output 1

Input 2 → Output 2

Input 3 → Output 3
```

This approach is suitable only when the input and output sequences have the same length.

# Limitations of Traditional RNN

Traditional RNNs face three major problems.

## 1. Variable-Length Sequences

Traditional RNNs assume:

```text
Input Length = Output Length
```

However, in tasks like machine translation, one sentence may translate into a shorter or longer sentence.

Example:

```text
English: I am hungry.
Nepali : मलाई भोक लागेको छ.

German : Übermorgen
English: The day after tomorrow
```

Therefore,

```text
Input Length ≠ Output Length
```

## 2. Vanishing Gradient

During training, gradients become very small for long sequences. As a result:

- Earlier information is forgotten.
- Long-term dependencies are difficult to learn.
- Training becomes less effective.

## 3. Limited Context

Traditional RNNs cannot remember important information from long sentences for a long time. They mainly focus on recent words, making them less effective for long sequences.

# Why Encoder–Decoder?

Many NLP tasks require the model to understand the entire input before producing the output. Examples include:

- Machine Translation
- Text Summarization
- Speech Recognition
- Image Captioning

To solve this problem, the **Encoder–Decoder** architecture was introduced.

The **Encoder** reads the complete input sentence and converts it into a **Context Vector**, which captures its meaning. The **Decoder** then uses this context vector to generate the output sentence one word at a time.

This architecture makes sequence-to-sequence tasks more accurate and efficient.

# Encoder–Decoder Architecture

The **Encoder–Decoder** architecture is a Sequence-to-Sequence (Seq2Seq) model that consists of two RNNs: an **Encoder** and a **Decoder**. It is mainly used for tasks like machine translation, text summarization, and speech recognition, where the input and output sequences may have different lengths.

The encoder reads the complete input sentence and converts it into a fixed-length representation called the **Context Vector**. The decoder then uses this context vector to generate the output sentence one word at a time.

```text
Input Sentence
      │
      ▼
  Encoder
      │
      ▼
Context Vector
      │
      ▼
  Decoder
      │
      ▼
Output Sentence
```

Unlike a traditional RNN, the Encoder–Decoder model can handle variable-length input and output sequences.

# Encoder

The **Encoder** is responsible for reading the entire input sentence one word at a time. As each word is processed, the hidden state is updated. After the last word, the final hidden state is produced, which summarizes the meaning of the whole sentence.

Example:

```text
I → love → cats
        │
        ▼
 Final Hidden State
(Context Vector)
```

### Role of the Encoder

- Reads the complete input sentence.
- Updates the hidden state at every time step.
- Learns the meaning of the sentence.
- Produces the **Context Vector**.
- Does not generate any output words.

### Encoder in Code

```python
embedded = self.embedding(input)
output, hidden = self.rnn(embedded)
```

- `embedded` contains the embedding vectors of the input words.
- `output` contains hidden states for all time steps.
- `hidden` is the final hidden state (Context Vector).

# Context Vector

The **Context Vector** is the encoder's final hidden state. It summarizes the important information from the entire input sentence and passes it to the decoder.

The better the context vector represents the sentence, the better the decoder can generate the correct output.

```text
Input Sentence
      │
      ▼
  Encoder
      │
      ▼
Context Vector
      │
      ▼
  Decoder
```

# Decoder

The **Decoder** generates the output sentence one word at a time using the context vector.

It starts with:

- The **Context Vector** as its initial hidden state.
- The **Start of Sentence (`<SOS>`)** token as its first input.

```text
<SOS>
   │
   ▼
Decoder
   │
   ▼
Word 1 → Word 2 → Word 3 → ... → <EOS>
```

The decoder stops generating words when it predicts the **End of Sentence (`<EOS>`)** token.

# Teacher Forcing

During training, the correct target word is given as the next input instead of the decoder's own prediction. This technique is called **Teacher Forcing**.

Example:

```text
Input        Output

<SOS>  →  I
I      →  am
am     →  happy
happy  →  <EOS>
```

### Advantages

- Faster training
- Stable learning
- Better accuracy during training

# Decoder During Inference

During testing, the correct target sentence is not available. Therefore, the decoder uses its own predicted word as the next input until it generates `<EOS>`.

# Why `topk(1)`?

The decoder predicts probabilities for all words in the vocabulary.

```python
_, topi = decoder_output.topk(1)
```

`topk(1)` selects the word with the highest probability, and its index is used as the next input.

# Why `detach()`?

```python
decoder_input = topi.squeeze(-1).detach()
```

`detach()` removes the predicted word from the computation graph so that PyTorch does not backpropagate through previous predictions. This saves memory and speeds up inference.

# Encoder vs Decoder

| Encoder | Decoder |
|---------|---------|
| Reads the input sentence. | Generates the output sentence. |
| Produces the Context Vector. | Uses the Context Vector to generate words. |
| Processes the whole sequence at once. | Generates one word at a time. |
| Does not generate output words. | Stops when `<EOS>` is predicted. |

# Code Comparison

| Encoder | Decoder |
|---------|---------|
| `output, hidden = self.rnn(embedded)` | `output, hidden = self.rnn(input, hidden)` |
| Entire input sequence is processed together. | One token is processed at each step. |
| Loop handled internally by PyTorch. | Loop written manually using `for`. |

# Summary

The Encoder–Decoder architecture overcomes the limitations of traditional RNNs by separating the task into two stages. The **Encoder** reads the entire input sentence and converts it into a **Context Vector**, while the **Decoder** uses this vector to generate the output sentence one word at a time. During training, **Teacher Forcing** improves learning, and during inference, the decoder relies on its own predictions until the `<EOS>` token is generated.

# Why Not Feed One-Hot Vectors Directly into an RNN?

A common question in Natural Language Processing (NLP) is:

> **If every word can be represented using a one-hot vector, why do we still need an embedding layer before feeding it into an RNN?**

Although an RNN can accept one-hot vectors, they are not efficient because they are large, sparse, and do not capture the meaning of words. An **Embedding Layer** solves these problems by converting words into smaller, meaningful vectors.

# What is a One-Hot Vector?

A **one-hot vector** is a binary representation of a word where only one position contains **1** and all other positions contain **0**.

Example vocabulary:

| Index | Word |
|------|------|
| 0 | hello |
| 1 | world |
| 2 | I |
| 3 | love |
| 4 | AI |

One-hot representation:

```text
hello → [1,0,0,0,0]

world → [0,1,0,0,0]

I → [0,0,1,0,0]

love → [0,0,0,1,0]

AI → [0,0,0,0,1]
```

Each word has its own unique position.

# Can We Feed One-Hot Vectors into an RNN?

Yes. An RNN can directly process one-hot vectors.

```text
Word
  │
  ▼
One-Hot Vector
  │
  ▼
 RNN
```

However, this approach is inefficient and creates several problems.

# Problems with One-Hot Vectors

## 1. Large Input Size

Real-world vocabularies can contain **50,000–100,000** words.

If the vocabulary has **100,000 words**, every word must be represented by a vector of length **100,000**.

```text
[0,0,0,...,1,...,0]
```

Most values are zero, making the vector **sparse**. This increases memory usage and slows down computation.

## 2. No Semantic Meaning

A one-hot vector only identifies a word. It does not tell the model how words are related.

For example,

```text
King

[1,0,0,0]

Queen

[0,1,0,0]
```

The model cannot understand that:

```text
King ≈ Queen

Dog ≈ Cat
```

Every word appears equally different from every other word.

## 3. Large Number of Parameters

Suppose,

```text
Vocabulary Size = 100,000

Hidden Size = 256
```

The RNN needs a weight matrix of:

```text
100,000 × 256
```

which contains about **25.6 million parameters**.

This results in:

- Higher memory usage
- Slower training
- More computation

# Embedding Layer

Instead of using one-hot vectors, modern NLP models use an **Embedding Layer**.

It converts each word into a small dense vector.

```text
Word
  │
  ▼
Word Index
  │
  ▼
Embedding Layer
  │
  ▼
Dense Vector
  │
  ▼
Encoder RNN
```

Example:

Instead of

```text
[0,0,0,1,0]
```

the embedding layer produces

```text
[0.21, 0.52, 0.18, 0.76]
```

These values are learned automatically during training.

# Advantages of Embeddings

Embedding vectors are:

- Small and dense
- Memory efficient
- Faster to process
- Learnable
- Able to capture word meaning

For example,

```text
King ≈ Queen

Dog ≈ Cat

Car ≈ Truck
```

Similar words have similar embedding vectors.

# Embedding Lookup

An embedding layer is mathematically the same as multiplying a one-hot vector by an embedding matrix.

```text
One-Hot Vector × Embedding Matrix

=

Embedding Lookup
```

Example:

| Word | Embedding |
|------|-----------|
| cat | [0.2, 0.5, 0.1] |
| dog | [0.3, 0.6, 0.2] |
| car | [0.8, 0.1, 0.7] |

For the word **dog**:

```text
One-Hot

[0,1,0]

↓

Embedding

[0.3,0.6,0.2]
```

Instead of creating the full one-hot vector, the embedding layer directly retrieves the required row, making the process much faster.

# Trainable Embeddings

Embedding vectors are initialized with random values.

Example:

```text
Dog

[0.13,-0.42,0.91]

Cat

[-0.22,0.51,0.34]
```

During training, these values are updated using backpropagation.

Words that appear in similar contexts gradually move closer together.

Example:

```text
The dog chased the ball.

The cat chased the mouse.
```

After training, the model learns:

```text
Dog ≈ Cat
```

# Why Use an Embedding Layer?

Although an RNN can learn directly from one-hot vectors, it is inefficient because:

- Input vectors are very large.
- Training is slower.
- Memory usage is high.
- More parameters are required.
- Word relationships must be learned from scratch.

An embedding layer first converts words into meaningful dense vectors, making learning faster and more effective.

# Complete NLP Pipeline

```text
Sentence

"I am Groot"

      │
      ▼

Word Tokenization

      │
      ▼

Word Indices

[12,45,891]

      │
      ▼

Embedding Layer

      │
      ▼

Dense Vectors

      │
      ▼

Encoder

      │
      ▼

Context Vector

      │
      ▼

Decoder

      │
      ▼

Predicted Sentence
```

| Stage | Purpose |
|--------|---------|
| Tokenization | Splits the sentence into words |
| Word Indexing | Converts words into IDs |
| Embedding Layer | Converts IDs into dense vectors |
| Encoder | Learns the input sentence representation |
| Context Vector | Stores the sentence meaning |
| Decoder | Generates the output sentence |



In [1]:
from __future__ import unicode_literals, print_function, division
from io import open
import unicodedata
import re
import random

import torch
import torch.nn as nn
from torch import optim
import torch.nn.functional as F

import numpy as np
from torch.utils.data import TensorDataset, DataLoader, RandomSampler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.set_default_device(device)
print(f"Using device = {torch.get_default_device()}")

Using device = cpu


In [2]:
SOS_token = 0 # Start of the Sentence
EOS_token = 1 # End of the Sentence

class Lang:
    def __init__(self, name):
        self.name = name
        self.word2index = {}
        self.word2count = {}
        self.index2word = {0: "SOS", 1: "EOS"}
        self.n_words = 2  # Count SOS and EOS

    def addSentence(self, sentence):
        for word in sentence.split(' '):
            self.addWord(word)

    def addWord(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.word2count[word] = 1
            self.index2word[self.n_words] = word
            self.n_words += 1
        else:
            self.word2count[word] += 1

In [3]:
# Turn a Unicode string to plain ASCII, thanks to
# https://stackoverflow.com/a/518232/2809427
def unicodeToAscii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
    )

# Lowercase, trim, and remove non-letter characters
def normalizeString(s):
    s = unicodeToAscii(s.lower().strip())
    s = re.sub(r"([.!?])", r" \1", s)
    s = re.sub(r"[^a-zA-Z!?]+", r" ", s)
    return s.strip()

In [4]:
def readLangs(path:str):
    lang1 = 'eng'; lang2 = 'fra'
    print("Reading lines...")

    # Read the file and split into lines
    lines = open(path, encoding='utf-8').\
        read().strip().split('\n')

    # Split every line into pairs and normalize (english to french)
    pairs = [[normalizeString(s) for s in l.split('\t')] for l in lines]

    # Reverse pairs: English-French -> French-English
    pairs = [list(reversed(p)) for p in pairs]

    # Input is French, output is English
    input_lang = Lang(lang2)
    output_lang = Lang(lang1)

    return input_lang, output_lang, pairs

In [5]:
MAX_LENGTH = 5

eng_prefixes = (
    "i am ", "i m ",
    "he is", "he s ",
    "she is", "she s ",
    "you are", "you re ",
    "we are", "we re ",
    "they are", "they re "
)

def filterPair(p):
    return len(p[0].split(' ')) < MAX_LENGTH and \
        len(p[1].split(' ')) < MAX_LENGTH and \
        p[1].startswith(eng_prefixes)


def filterPairs(pairs):
    return [pair for pair in pairs if filterPair(pair)]

In [6]:
def prepareData(path):
    input_lang, output_lang, pairs = readLangs(path)
    print("Read %s sentence pairs" % len(pairs))
    pairs = filterPairs(pairs)
    print("Trimmed to %s sentence pairs" % len(pairs))
    print("Counting words...")
    for pair in pairs:
        input_lang.addSentence(pair[0])
        output_lang.addSentence(pair[1])
    print("Counted words:")
    print(input_lang.name, input_lang.n_words)
    print(output_lang.name, output_lang.n_words)
    return input_lang, output_lang, pairs

In [7]:
PATH = r'eng-fra.txt'

input_lang, output_lang, pairs = prepareData(PATH)
print(random.choice(pairs))

output_lang.word2index['am']  # try different English words.

Reading lines...
Read 135842 sentence pairs
Trimmed to 3272 sentence pairs
Counting words...
Counted words:
fra 1757
eng 967
['je suis abasourdi', 'i m flabbergasted']


15

In [8]:
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size, dropout_p=0.1):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size

        self.embedding = nn.Embedding(input_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, input):
        embedded = self.dropout(self.embedding(input))
        output, hidden = self.rnn(embedded)
        return output, hidden

In [9]:
class DecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size):
        super(DecoderRNN, self).__init__()
        self.embedding = nn.Embedding(output_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        self.out = nn.Linear(hidden_size, output_size)

    def forward(self, encoder_outputs, encoder_hidden, target_tensor=None):
        batch_size = encoder_outputs.size(0)
        decoder_input = torch.empty(batch_size, 1, dtype=torch.long, device=device).fill_(SOS_token)
        decoder_hidden = encoder_hidden
        decoder_outputs = []

        for i in range(MAX_LENGTH):
            decoder_output, decoder_hidden  = self.forward_step(decoder_input, decoder_hidden)
            decoder_outputs.append(decoder_output)

            if target_tensor is not None:
                # Teacher forcing: Feed the target as the next input
                decoder_input = target_tensor[:, i].unsqueeze(1) # Teacher forcing
            else:
                # Without teacher forcing: use its own predictions as the next input
                _, topi = decoder_output.topk(1) # values, index of the highest-scoring word
                decoder_input = topi.squeeze(-1).detach()  # detach from history as input (removes the last dimension before detaching)

        decoder_outputs = torch.cat(decoder_outputs, dim=1)
        decoder_outputs = F.log_softmax(decoder_outputs, dim=-1)
        return decoder_outputs, decoder_hidden, None # We return `None` for consistency in the training loop

    def forward_step(self, input, hidden):
        output = self.embedding(input)
        output = F.relu(output)
        output, hidden = self.rnn(output, hidden)
        output = self.out(output)
        return output, hidden

In [10]:
def indexesFromSentence(lang, sentence):
    return [lang.word2index[word] for word in sentence.split(' ')]

def tensorFromSentence(lang, sentence):
    indexes = indexesFromSentence(lang, sentence)
    indexes.append(EOS_token)
    return torch.tensor(indexes, dtype=torch.long, device=device).view(1, -1)

def tensorsFromPair(pair):
    input_tensor = tensorFromSentence(input_lang, pair[0])
    target_tensor = tensorFromSentence(output_lang, pair[1])
    return (input_tensor, target_tensor)

def get_dataloader(batch_size):
    input_lang, output_lang, pairs = prepareData(path=PATH)

    n = len(pairs)
    input_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)
    target_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)

    for idx, (inp, tgt) in enumerate(pairs):
        inp_ids = indexesFromSentence(input_lang, inp)
        tgt_ids = indexesFromSentence(output_lang, tgt)
        inp_ids.append(EOS_token)
        tgt_ids.append(EOS_token)
        input_ids[idx, :len(inp_ids)] = inp_ids
        target_ids[idx, :len(tgt_ids)] = tgt_ids

    train_data = TensorDataset(torch.LongTensor(input_ids).to(device),
                               torch.LongTensor(target_ids).to(device))

    train_sampler = RandomSampler(train_data)
    train_dataloader = DataLoader(train_data, sampler=train_sampler, batch_size=batch_size)
    return input_lang, output_lang, train_dataloader

In [11]:
def train_epoch(dataloader, encoder, decoder, encoder_optimizer,
          decoder_optimizer, criterion):

    total_loss = 0
    for data in dataloader:
        input_tensor, target_tensor = data

        encoder_optimizer.zero_grad()
        decoder_optimizer.zero_grad()

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, _, _ = decoder(encoder_outputs, encoder_hidden, target_tensor) # using teacher forcing

        loss = criterion(
            decoder_outputs.view(-1, decoder_outputs.size(-1)),
            target_tensor.view(-1)
        )
        loss.backward()

        encoder_optimizer.step()
        decoder_optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

In [12]:
import time
import math

def asMinutes(s):
    m = math.floor(s / 60)
    s -= m * 60
    return '%dm %ds' % (m, s)

def timeSince(since, percent):
    now = time.time()
    s = now - since
    es = s / (percent)
    rs = es - s
    return '%s (- %s)' % (asMinutes(s), asMinutes(rs))

In [13]:
import matplotlib.pyplot as plt
plt.switch_backend('agg')
import matplotlib.ticker as ticker
import numpy as np

def showPlot(points):
    plt.figure()
    fig, ax = plt.subplots()
    # this locator puts ticks at regular intervals
    loc = ticker.MultipleLocator(base=0.2)
    ax.yaxis.set_major_locator(loc)
    plt.plot(points)

In [14]:
def train(train_dataloader, encoder, decoder, n_epochs, learning_rate=0.001,
               print_every=100, plot_every=100):
    start = time.time()
    plot_losses = []
    print_loss_total = 0  # Reset every print_every
    plot_loss_total = 0  # Reset every plot_every

    encoder_optimizer = optim.Adam(encoder.parameters(), lr=learning_rate)
    decoder_optimizer = optim.Adam(decoder.parameters(), lr=learning_rate)
    criterion = nn.NLLLoss()

    for epoch in range(1, n_epochs + 1):
        loss = train_epoch(train_dataloader, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion)
        print_loss_total += loss
        plot_loss_total += loss

        if epoch % print_every == 0:
            print_loss_avg = print_loss_total / print_every
            print_loss_total = 0
            print('%s (%d %d%%) %.4f' % (timeSince(start, epoch / n_epochs),
                                        epoch, epoch / n_epochs * 100, print_loss_avg))

        if epoch % plot_every == 0:
            plot_loss_avg = plot_loss_total / plot_every
            plot_losses.append(plot_loss_avg)
            plot_loss_total = 0

    showPlot(plot_losses)

In [15]:
def evaluate(encoder, decoder, sentence, input_lang, output_lang):
    with torch.no_grad():
        input_tensor = tensorFromSentence(input_lang, sentence)

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, decoder_hidden, decoder_attn = decoder(encoder_outputs, encoder_hidden)

        _, topi = decoder_outputs.topk(1)
        decoded_ids = topi.squeeze()

        decoded_words = []
        for idx in decoded_ids:
            if idx.item() == EOS_token:
                decoded_words.append('<EOS>')
                break
            decoded_words.append(output_lang.index2word[idx.item()])
    return decoded_words, decoder_attn

In [16]:
def evaluateRandomly(encoder, decoder, n=10):
    for i in range(n):
        pair = random.choice(pairs)
        print('>', pair[0])
        print('=', pair[1])
        output_words, _ = evaluate(encoder, decoder, pair[0], input_lang, output_lang)
        output_sentence = ' '.join(output_words)
        print('<', output_sentence)
        print('')

In [ ]:
hidden_size = 128
batch_size = 32
EPOCHS = 200

input_lang, output_lang, train_dataloader = get_dataloader(batch_size)

encoder = EncoderRNN(input_lang.n_words, hidden_size).to(device)
decoder = DecoderRNN(hidden_size, output_lang.n_words).to(device)

train(train_dataloader, encoder, decoder, EPOCHS, print_every=5, plot_every=5)

Reading lines...
Read 135842 sentence pairs
Trimmed to 3272 sentence pairs
Counting words...
Counted words:
fra 1757
eng 967
0m 6s (- 4m 19s) (5 2%) 1.9671
0m 10s (- 3m 28s) (10 5%) 1.2930
0m 15s (- 3m 14s) (15 7%) 1.0926
0m 20s (- 3m 4s) (20 10%) 0.9456
0m 24s (- 2m 53s) (25 12%) 0.8193
0m 30s (- 2m 55s) (30 15%) 0.7138


In [ ]:
encoder.eval()
decoder.eval()
evaluateRandomly(encoder, decoder)

> nous en avons termine
= we re all done
< we re finished <EOS>

> je suis serieuse
= i m not bluffing
< i am pretty good <EOS>

> je viens de portugal
= i am from portugal
< i am from portugal <EOS>

> il est pauvre
= he is poor
< they re jittery <EOS>

> tu es riche
= you re rich
< you are entirely correct <EOS>

> vous etes tres raffine
= you re very sophisticated
< you re very sophisticated <EOS>

> tu es libre
= you re free
< you are welcome here <EOS>

> ce sont des bebes
= they re babies
< they re babies <EOS>

> vous en avez termine
= you re through
< you re through <EOS>

> nous rejouons
= we re taking over
< we re really good <EOS>



# difference between encoder and decoder 


| Encoder                                    | Decoder                                          |
| ------------------------------------------ | ------------------------------------------------ |
| `self.rnn(embedded)`                       | `self.rnn(output, hidden)`                       |
| Whole sentence goes into RNN at once       | One token goes into RNN each time                |
| No `for` loop in your code                 | `for i in range(MAX_LENGTH)` loop                |
| Reads input sequence                       | Generates output sequence                        |
| Returns final hidden state                 | Uses previous hidden state to generate next word |
| No prediction layer needed during encoding | Uses `self.out(output)` to predict the next word |


# Discussion

The old way of doing Recurrent Neural Networks is good for jobs where each thing you put in has a matching thing that comes out like figuring out what part of speech a word is and finding the names of people and places.. This old way has trouble with jobs where the thing you put in and the thing that comes out are different lengths. It also has trouble remembering things from a time ago because the information gets lost as it goes through the system.

The Encoder and Decoder system is a way to fix these problems. It breaks the job into two parts. The Encoder looks at all of the thing you put in. Turns it into a special set of numbers that it calls the Context Vector. Then the Decoder uses these numbers to make the thing that comes out one word at a time. This way the system can handle things that're different lengths.

Another good thing is the Embedding Layer. Of using big lists of numbers that are mostly zeros the Embedding Layer turns words into small lists of numbers that mean something. This uses memory makes the system train faster and helps it understand what words are similar.

So when you put the Embedding Layer with the Encoder and Decoder system it makes the system work a lot better for jobs like translating languages summarizing text and recognizing speech.

#

The RNN Encoder and Decoder system is an improvement over the old way. It can handle things that're different lengths. The Encoder looks at the thing you put in and tries to understand it. The Decoder makes the thing that comes out one step at a time. The Embedding Layer makes it even better, by turning lists of numbers into small ones that mean something. All of these things together make the system work efficiently and accurately and that is why it is used in a lot of modern Natural Language Processing models.